# Lektion 09 - Metakognitions designmønster


## Opsætning

Denne notesbog demonstrerer Metakognition designmønsteret ved hjælp af Microsoft Agent Framework.

**Forudsætninger:**
- Azure OpenAI-implementering konfigureret via miljøvariabler
- Azure CLI autentificeret (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Hvad er metakognition?

Metakognition er **at tænke over tænkning**. I konteksten af AI-agenter betyder det at bygge agenter, der kan:

- **Selvreflektere** over deres egne output og ræsonnementer
- **Optræde fejl** og komme sig yndefuldt i stedet for at fejle tavst
- **Vurdere** om deres svar er komplette og hjælpsomme
- **Tilpasse** deres strategi, når en indledende tilgang ikke virker (f.eks. at falde tilbage til et backup-system)

En metakognitiv agent svarer ikke bare på spørgsmål — den overvåger sin egen præstation og justerer undervejs.


## Primære og backup-værktøjer

Et almindeligt metakognitionsmønster er **fallback-strategien**. Agenten prøver først et primært værktøj; hvis det fejler (fx en 404-fejl), genkender agenten fejlen og skifter gennemsigtigt til et backup-værktøj.

Dette afspejler virkelige systemer, hvor primære tjenester kan være utilgængelige, og agenter skal selv diagnosticere problemet, inden de vælger en alternativ vej.

Nedenfor definerer vi to værktøjer til flysøgning:
- **Primær** — dækker Paris, Tokyo og Barcelona
- **Backup** — dækker Berlin, Sydney og New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Selvroterende agent med fejlgendannelse

Agenten nedenfor er instrueret i først at prøve det primære flyvesystem, genkende fejl og transparent falde tilbage til backupsystemet. Efter hvert svar evaluerer den kort sig selv for, om den fuldt ud besvarede brugerens spørgsmål.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Mønster for selvevaluering

En anden facet af metakognition er **selvevaluering**: en separat agent (eller den samme agent i et andet gennemløb) gennemgår et svar for fuldstændighed, nøjagtighed og hjælpsomhed.

Nedenfor opretter vi en `ResponseEvaluator`-agent, der vurderer rejseagenters svar på tre dimensioner.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Resumé

I denne lektion lærte du, hvordan man bygger **metakognitive agenter** ved hjælp af Microsoft Agent Framework:

- **Selvrefleksion**: Agenter, der overvåger deres egen ræsonnering og åbent kommunikerer, hvad der skete.
- **Fejlrettelse med fallback**: Et primært + backup værktøjsmønster, hvor agenten opdager fejl (f.eks. 404 fejl) og automatisk prøver en alternativ kilde.
- **Selvevaluering**: En separat evalueringsagent, der scorer svar for fuldstændighed, nøjagtighed og hjælpsomhed.

Disse mønstre gør agenter mere robuste, gennemsigtige og pålidelige – vigtige egenskaber for produktionsimplementeringer.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
